In [1]:
import os
from ast import literal_eval
import pandas as pd
import spacy
import re
from spacy.tokenizer import Tokenizer
import matplotlib.pyplot as plt
import dill
import numpy as np
from typing import Optional, List, Tuple

In [2]:
def print_out_clickable_link(df):
    for index, row in df.iterrows():
        print(f"Index: {index}, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix={row['object_key']}")

In [3]:
data = pd.read_csv('../../data/PFUB_ERLASS_dataset_processed_with_geschaeftsnummer.csv')

In [4]:
data = data[data['is_pfub']==True].reset_index(drop=True)

In [5]:
data

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags
0,Amtsgericht Memmingen\nAbteilung für Zwangsvol...,01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...,approved_attachment_and_transfer_order,amtsgericht memmingen\nabteilung für zwangsvol...,"{'case_slug': '163464253141', 'debtor_name': '...",True,False,"['amtsgericht', 'memmingen', 'abteilung', 'zwa...",amtsgericht memmingen\nabteilung für zwangsvol...
1,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,01.06.2022/1654077659_Doc_01062022_000000001_1...,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '110872558124', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...
2,Amtsgericht Eschweiler\n070071 967072\n-Geschä...,01.06.2022/1654077662_Doc_01062022_000000001_1...,approved_seizure,amtsgericht eschweiler\n070071 967072\n-geschä...,"{'case_slug': '114249164962', 'debtor_name': '...",True,False,"['amtsgericht', 'eschweiler', '-geschäftsstell...",amtsgericht eschweiler\n<NUMBER><PHONE>\n-gesc...
3,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,"['amtsgericht', 'münster', 'amtsgericht', 'mün...",amtsgericht münster\n<NUMBER>- amtsgericht mün...
4,Amtsgericht Viersen\n-Geschäftsstelle-\n-15- A...,01.07.2025/ocr-v2_Allgemeines_Schreiben_R__202...,approved_attachment_and_transfer_order,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,"{'case_slug': '125495503778', 'debtor_name': '...",True,False,"['amtsgericht', 'viersen', '-geschäftsstelle-'...",amtsgericht viersen\n-geschäftsstelle-\n<NUMBE...
...,...,...,...,...,...,...,...,...,...
463,Amtsgericht Wuppertal\n-44- Amtsgericht Wupper...,31.01.2023/1675166710_YA_003_31012023_124814.pdf,approved_seizure,amtsgericht wuppertal\n-44- amtsgericht wupper...,"{'case_slug': '110454161073', 'debtor_name': '...",True,False,"['amtsgericht', 'wuppertal', 'amtsgericht', 'w...",amtsgericht wuppertal\n<NUMBER>- amtsgericht w...
464,"Amtsgericht Itzehoe\nAmtsgericht Itzehoe, Berg...",31.01.2023/1675166710_YA_003_31012023_124815.pdf,approved_seizure,"amtsgericht itzehoe\namtsgericht itzehoe, berg...","{'case_slug': None, 'debtor_name': None, 'cred...",True,False,"['amtsgericht', 'itzehoe', 'amtsgericht', 'itz...","amtsgericht itzehoe\namtsgericht itzehoe, berg..."
465,Amtsgericht Leverkusen\n-45- Amtsgericht Lever...,31.05.2023/1685534803_PF_005_31052023_140508.pdf,approved_seizure,amtsgericht leverkusen\n-45- amtsgericht lever...,"{'case_slug': '117578968598', 'debtor_name': '...",True,False,"['amtsgericht', 'leverkusen', 'amtsgerichen', ...",amtsgericht leverkusen\n<NUMBER>- amtsgericht ...
466,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,31.07.2023/1690794316_KD_004_31072023_110320.pdf,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '125677855012', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...


In [6]:
different = data['data'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
keys_set = set()
for i in range(len(different)):
    if isinstance(different[i], dict):
        keys = different[i].keys()
        keys_set.update(keys)
keys_set # No date data.

{'case_slug',
 'creditor_name',
 'debtor_name',
 'document_type',
 'recognition_errors'}

In [7]:
# USE CURRENT DATE EXTRACTOR FOR EXTRACTION
import os


notebook_dir = os.getcwd()
project_dir = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(notebook_dir))))
print("Notebook Directory:", notebook_dir)
print("Project Directory:", project_dir)
os.chdir(project_dir)

Notebook Directory: /Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court/pfub-erlass/param_extraction
Project Directory: /Users/melih.gorgulu/Desktop/Projects/intent_recognition


In [8]:
from src.services.attachment_processing.aftercourt_extractors.ladung.summon_date_extractor import DateExtractor
import dateparser

class PFUBDateExtractor(DateExtractor):
    def __init__(self):
        super().__init__()
    
    def extract(self, text: str):
        date_matches = self._extract_dates_with_regex(text)
        valid_date_matches = []
        for d in date_matches:
            if self.is_valid_date(d[0]):
                valid_date_matches.append(d)

        if not valid_date_matches:
            return None
        
        return valid_date_matches
    
    def is_valid_date(self, match_date_text:str):
        parsed_date = dateparser.parse(match_date_text, languages=['de'])
        if parsed_date is None:
            return False
        
        if '.' not in match_date_text:
            # Check UTC, it should be 00:00:00
            if parsed_date.hour != 0 or parsed_date.minute != 0 or parsed_date.second != 0:
                return False
        
        if parsed_date.year < 2000:
            return False
        return True
        

date_extractor = PFUBDateExtractor()

In [9]:
data['regex_date_extracted'] = data['cleaned_text'].apply(lambda x: date_extractor.extract(x))
data['n_of_dates'] = data['regex_date_extracted'].apply(lambda x: len(x) if x else 0)

In [10]:
check = data[['object_key','cleaned_text', 'regex_date_extracted','n_of_dates']]

In [11]:
check_nones = check[check['n_of_dates']==0]
check_nones

,object_key,cleaned_text,regex_date_extracted,n_of_dates
427,29.06.2023/1688029576_KD_004_29062023_110207.pdf,amtsgericht wuppertal\n-44- amtsgericht wupper...,None,0


In [12]:
check['n_of_dates'].value_counts()

n_of_dates
2    336
1     80
3     45
4      3
5      2
9      1
0      1
Name: count, dtype: int64

In [13]:
check_big_nums = check[check['n_of_dates']>3]
check_big_nums

,object_key,cleaned_text,regex_date_extracted,n_of_dates
57,04.12.2024/ocr-v2_RB_008_04122024_133334.pdf,amtsgericht wedding\n-zentrales mahngericht be...,"[(25.11.2024, 115, 125), (3. dez. 2024, 634, 6...",5
101,07.07.2023/1688724321_KD_003_07072023_114443.pdf,amtsgericht duisburg\n-631- amtsgericht duisbu...,"[(28.06.2023, 79, 89), (06. juli 2023, 223, 23...",5
268,19.09.2022/1663601443_Scan_YA_00519092022_1702...,amtsgericht flensburg\n4070071967072\namtsgeri...,"[(19. sep. 2022, 211, 224), (14.09.2022, 321, ...",4
289,21.06.2023/1687334835_KD_001_21062023_093715.pdf,amtsgericht euskirchen\npair finance gmbi\n19....,"[(19. juni 2023, 41, 54), (09.06.2023, 112, 12...",4
365,25.11.2022/1669374367_Scan_AK_003_17112022_114...,amtsgericht euskirchen\n4070071 967072\npair f...,"[(16. nov. 2022, 56, 69), (10.11.2022, 125, 13...",9
389,27.06.2023/1687860404_KD_002_27062023_115313.pdf,amtsgericht münster\n-33- amtsgericht münster ...,"[(13.06.2023, 75, 85), (26. juni 2023, 232, 24...",4


In [14]:
print_out_clickable_link(check_big_nums)

Index: 57, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=04.12.2024/ocr-v2_RB_008_04122024_133334.pdf
Index: 101, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=07.07.2023/1688724321_KD_003_07072023_114443.pdf
Index: 268, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=19.09.2022/1663601443_Scan_YA_00519092022_170213.pdf
Index: 289, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=21.06.2023/1687334835_KD_001_21062023_093715.pdf
Index: 365, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=25.11.2022/1669374367_Scan_AK_003_17112022_114645.pdf
Index: 389, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=27.06.2023/1687860404_KD_002_27062023_115313.pdf


In [15]:
check_big_nums.iloc[0]['regex_date_extracted']

[('25.11.2024', 115, 125),
 ('3. dez. 2024', 634, 646),
 ('04.11.2024', 803, 813),
 ('19.06.2024', 880, 890),
 ('10.07.2024', 913, 923)]

In [16]:
def print_out_extracted_date_with_colorful_text(txt, dates):
    if not dates:
        print(txt)
        return
    
    # ANSI color codes
    RED = '\033[91m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    BLUE = '\033[94m'
    MAGENTA = '\033[95m'
    CYAN = '\033[96m'
    RESET = '\033[0m'
    BOLD = '\033[1m'
    
    colors = [RED, GREEN, YELLOW, BLUE, MAGENTA, CYAN]
    
    # Sort dates by start index
    sorted_dates = sorted(dates, key=lambda x: x[1])
    
    result = ""
    last_end = 0
    
    for i, (date_str, start_idx, end_idx) in enumerate(sorted_dates):
        # Add text before the date
        result += txt[last_end:start_idx]
        # Add colored date
        color = colors[i % len(colors)]
        result += f"{BOLD}{color}{date_str}{RESET}"
        last_end = end_idx
    
    # Add remaining text
    result += txt[last_end:]
    
    print(result)

In [17]:

idx = 0
data_investigate = check[check['n_of_dates']==9].iloc[idx]
#data_investigate = check_big_nums.iloc[idx]
dates = data_investigate['regex_date_extracted']
text = data_investigate['cleaned_text']
object_key = data_investigate['object_key']
s3_link = F"https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix={object_key}"
n_of_dates = data_investigate['n_of_dates']

print(F"EXTRACTED DATES HIGHLIGHTED:")
print("Number of extracted dates:", n_of_dates)
print("S3:",s3_link )

print("-"*100)

print_out_extracted_date_with_colorful_text(text, dates)

EXTRACTED DATES HIGHLIGHTED:
Number of extracted dates: 9
S3: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix=25.11.2022/1669374367_Scan_AK_003_17112022_114645.pdf
----------------------------------------------------------------------------------------------------
amtsgericht euskirchen
4070071 967072
pair finance gmbh
16. nov. 2022
-401- amtsgericht euskirchen postfach 53877 euskirchen
10.11.2022
seite 1 von 1
pair finance gmbh
aktenzeichen
hardenbergstraße 32
401 m 22/22
10623 berlin
bei antwort bitte angeben
bearbeiter
frau decker
durchwahl
02251/951-2460
ihr zeichen: 14389/22sk01 /sk 107422363534
sehr geehrte damen und herren,
in der zwangsvollstreckungssache
liquandum capital gmbh gegen bung
wurde ihrem antrag auf erlass eines pfändungs- und
überweisungsbeschlusses entsprochen. anbei titel 2urture
die zustellung des pfändungs- und überweisungsbeschluss an
anschrift
schuldner und drittschuldner ist veranlasst worden.
kölner str. 40-42
5

In [18]:
# NOTES:
# 1) For # of date=3, we usually have document date on top right, 1 date from pair finance stamp, 1 date from erlass date [TARGET]
#    -> for this we can use keyword scoring
# 2) For # of date=2, we usually have document date on top right, 1 date from pair finance stamp
# -> TODO: We need to know for this cases if the document date is target. If yes, then we can eliminate date from stamp using
# -> its surroindundings (such as it always has "PAIR Finance" word araound it)
# 3) For dates more than 3, sometimes the regex extracts wrong dates. But they can be corrected later in validation step.
# 4) some high number of dates extraction is comes from copy of the document. They sometime have exact copy of the document below.

In [19]:
def get_context_around_dates(text: str, date_matches: List[Tuple[str, int, int]], window_char: int =  40):
    if not date_matches:
        return None
    
    contexts = []
    date_text_unique = set()
    for date_match in date_matches:
        date_text, date_start, date_end = date_match
        if date_text in date_text_unique:
            continue
        else:
            date_text_unique.add(date_text)
            window_start = max(0, date_start - window_char)
            window_end = min(len(text), date_end + window_char)
            context = text[window_start: window_end]
            contexts.append(context)
    return contexts

In [20]:
data['context_date'] = data.apply(lambda row: get_context_around_dates(row['cleaned_text'], row['regex_date_extracted']), axis=1)

In [21]:
check = data[['n_of_dates', 'regex_date_extracted','context_date']]
check

,n_of_dates,regex_date_extracted,context_date
0,1,"[(31.03.2025, 593, 603)]",[zeichen\ndatum\n163464253141\n54 m 1067/25\n3...
1,2,"[(31. mai 2022, 230, 242), (27.05.2022, 323, 3...",[lefax:\n(069)1367 8767\npair finance gmbh\n31...
2,2,"[(25.05.2022, 114, 124), (31. mai 2022, 193, 2...",[chweiler postfach 1340 52233 eschweiler\n25.0...
3,3,"[(19.06.2024, 77, 87), (28. juni 2024, 235, 24...",[münster - postfach 6165 - 48136 münster\n19.0...
4,2,"[(11.06.2025, 93, 103), (30 -\n12.00, 1041, 10...",[t viersen postfach 100161 41701 viersen\n11.0...
...,...,...,...
463,1,"[(25.01.2023, 66, 76)]","[ amtsgericht wuppertal, 42097 wuppertal\n25.0..."
464,1,"[(23.01.2023, 350, 360)]",[1/20sk01 / sk 107009849608\n25 m 2307/22\n23....
465,2,"[(13.04.2023, 86, 96), (30. mai 2023, 254, 266)]","[usen, gerichtsstr. 9, 51379 leverkusen-\n13.0..."
466,2,"[(28. juli 2023, 235, 248), (25.07.2023, 311, ...",[mbh\npair finance gmbh\nhardenbergstr. 32\n28...


In [22]:
check_3 = check[check['n_of_dates']==3]
check_3

,n_of_dates,regex_date_extracted,context_date
3,3,"[(19.06.2024, 77, 87), (28. juni 2024, 235, 24...",[münster - postfach 6165 - 48136 münster\n19.0...
12,3,"[(20.12.2023, 77, 87), (02. jan. 2024, 235, 24...",[münster - postfach 6165 - 48136 münster\n20.1...
24,3,"[(25.06.2025, 73, 83), (01. juli 2025, 192, 20...",[cht münster postfach 6165 48136 münster\n25.0...
30,3,"[(31.08.2023, 73, 83), (29. sep. 2023, 219, 23...",[cht münster postfach 6165 48136 münster\n31.0...
33,3,"[(31. märz 2023, 37, 50), (23.03.2023, 106, 11...",[amtsgericht borken\npair finance gmbh\n31. mä...
39,3,"[(27.01.2025, 73, 83), (03. feb. 2025, 218, 23...",[cht münster postfach 6165 48136 münster\n27.0...
40,3,"[(21.02.2025, 75, 85), (03. märz 2025, 232, 24...",[t münster postfach 6165 - 48136 münster\n21.0...
54,3,"[(27.08.2025, 75, 85), (03. sep. 2025, 223, 23...",[t münster - postfach 6165 48136 münster\n27.0...
63,3,"[(17.05.2023, 75, 85), (02. juni 2023, 233, 24...",[t münster postfach 6165 - 48136 münster\n17.0...
78,3,"[(17.05.2023, 75, 85), (02. juni 2023, 233, 24...",[t münster postfach 6165 - 48136 münster\n17.0...
